In [84]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [85]:
from fastapi import FastAPI, File, HTTPException, Query, UploadFile
from fastapi.responses import StreamingResponse

In [86]:
from pypdfium2 import pypdfium2

In [87]:
import ollama

In [88]:
from langchain.chat_models import init_chat_model

In [89]:
KNOWLEDGE_BASE = """
1.1 WAKE UP AT A FIXED TIME
Set a consistent wake-up time and keep it every day, including weekends. A stable schedule trains your body clock so you wake feeling rested instead of groggy. Place the alarm across the room so you must physically get out of bed to switch it off. Avoid hitting snooze, as the short fragmented sleep it gives is low quality and leaves you more tired. The moment your feet touch the floor, the day has started.

1.2 GET SUNLIGHT EARLY
Within the first 30 minutes of waking, expose yourself to natural daylight. Open the curtains, step onto a balcony, or take a short walk outside. Morning light signals your brain to stop producing the sleep hormone melatonin and helps set your circadian rhythm for the day. This single habit improves alertness in the morning and makes it easier to fall asleep at night.

1.3 HYDRATE BEFORE CAFFEINE
You lose water through breathing and sweating during the night, so you wake up mildly dehydrated. Drink a full glass of water before reaching for coffee or tea. Hydration restores focus, reduces headaches, and kick-starts your metabolism. If you want, add a pinch of salt or a squeeze of lemon to help your body absorb the water. Delay caffeine by 60 to 90 minutes to avoid an early energy crash.

1.4 MOVE YOUR BODY
Spend 5 to 15 minutes on light physical activity such as stretching, yoga, a brisk walk, or a few bodyweight exercises. Movement increases blood flow, raises your core temperature, and releases endorphins that lift your mood. You do not need an intense workout; the goal is to wake the body gently and shake off stiffness from sleeping. Consistency matters far more than intensity.

1.5 EAT A BALANCED BREAKFAST
Fuel your body with a breakfast that combines protein, healthy fats, and fibre to keep energy steady until lunch. Good options include eggs, oats, yoghurt with fruit, or whole-grain toast with nut butter. Avoid sugary cereals and pastries, which spike blood sugar and lead to a mid-morning slump. Eating well in the morning improves concentration and reduces unhealthy snacking later in the day.

1.6 PLAN YOUR DAY
Take five quiet minutes to review your schedule and choose the three most important tasks for the day. Writing them down clears mental clutter and gives you a clear direction before distractions arrive. Tackle the hardest or most important task first, while your willpower and focus are at their peak. A small amount of planning prevents the day from running away from you.

1.7 AVOID YOUR PHONE FIRST
Resist the urge to check messages, email, or social media the moment you wake. Starting the day reacting to other people's demands puts you in a stressed, scattered state. Give yourself at least the first 30 minutes phone-free so your mind can settle and you can act on your own priorities. The notifications will still be there once your morning routine is done.
"""

In [90]:
chunks = KNOWLEDGE_BASE.strip().split("\n\n")
print(len(chunks))
chunks

7


['1.1 WAKE UP AT A FIXED TIME\nSet a consistent wake-up time and keep it every day, including weekends. A stable schedule trains your body clock so you wake feeling rested instead of groggy. Place the alarm across the room so you must physically get out of bed to switch it off. Avoid hitting snooze, as the short fragmented sleep it gives is low quality and leaves you more tired. The moment your feet touch the floor, the day has started.',
 '1.2 GET SUNLIGHT EARLY\nWithin the first 30 minutes of waking, expose yourself to natural daylight. Open the curtains, step onto a balcony, or take a short walk outside. Morning light signals your brain to stop producing the sleep hormone melatonin and helps set your circadian rhythm for the day. This single habit improves alertness in the morning and makes it easier to fall asleep at night.',
 '1.3 HYDRATE BEFORE CAFFEINE\nYou lose water through breathing and sweating during the night, so you wake up mildly dehydrated. Drink a full glass of water b

In [91]:
vectorize = TfidfVectorizer()
chunks_vectors = vectorize.fit_transform(chunks)
chunks_vectors

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 390 stored elements and shape (7, 292)>

In [92]:
def retrieve(query: str, k: int = 6):
    query_vector = vectorize.transform([query])
    similarity = cosine_similarity(query_vector, chunks_vectors).flatten()
    top_k_indices = np.argsort(similarity)[::-1][:k]
    return [chunks[i] for i in top_k_indices], similarity[top_k_indices]

In [93]:
def main():
    example_query = [
        "When should I wake up?",
        "Should I check my phone in the morning?",
        "What should I eat for breakfast?",
    ]

    for query in example_query:
        retrieve_chunks, scores = retrieve(query, k=2)

        print("=" * 70)
        print(f"QUERY: {query}")
        print("=" * 70)
        for rank, (chunk, score) in enumerate(zip(retrieve_chunks, scores), 1):
            preview = chunk[:200] + "..." if len(chunk) > 200 else chunk
            print(f"{rank} score={score:.3f}")
            print(preview)
            print("-" * 70)
        print()

main()

QUERY: When should I wake up?
1 score=0.283
1.1 WAKE UP AT A FIXED TIME
Set a consistent wake-up time and keep it every day, including weekends. A stable schedule trains your body clock so you wake feeling rested instead of groggy. Place the al...
----------------------------------------------------------------------
2 score=0.126
1.3 HYDRATE BEFORE CAFFEINE
You lose water through breathing and sweating during the night, so you wake up mildly dehydrated. Drink a full glass of water before reaching for coffee or tea. Hydration r...
----------------------------------------------------------------------

QUERY: Should I check my phone in the morning?
1 score=0.355
1.7 AVOID YOUR PHONE FIRST
Resist the urge to check messages, email, or social media the moment you wake. Starting the day reacting to other people's demands puts you in a stressed, scattered state. G...
----------------------------------------------------------------------
2 score=0.189
1.2 GET SUNLIGHT EARLY
Within the first 3

In [94]:
model = "qwen3-vl:4b"
TEMPERATURE = 0.0

In [95]:
RAG_PROMPT = """Use the following context to answer the question. If you cannot find the answer in the context say "I can't find the answer to this in the given context"

<contexts>
{context}
</contexts>

<question>
{query}
</question>
""".strip()

In [96]:
resp = ollama.chat(model=model, messages=[
    {"role": "user", "content": "Explain RAG in one line"}
])
print(resp["message"]["content"])

**Retrieval-Augmented Generation (RAG)** is an AI technique that **retrieves relevant documents from a knowledge base to augment the generation of more accurate, context-aware responses**.  

*(One line, concise, and captures the core function: retrieval + generation for improved accuracy.)*


In [97]:
llm = init_chat_model(
    model=model,
    model_provider="ollama",
    temperature=TEMPERATURE,
    reasoning=False
)

In [98]:
def answer(query: str, k: int = 2):
    context_chunks, scores = retrieve(query, k)
    context_string = "\n\n".join(
        f"<context>\n{c}\n</context>" for c in context_chunks
    )
    prompt = RAG_PROMPT.format(context=context_string, query=query)
    for chunk in llm.stream(prompt):
        yield chunk.content

In [99]:
def ask_question(query: str):
    """Stream the answer with plain print. Strips <think>...</think> reasoning."""
    full_response = ""
    in_thinking = False

    print(f"Q: {query}\n")
    print("A: ", end="", flush=True)
    for content in answer(query):
        full_response += content

        if "<think>" in content:
            in_thinking = True
            content = content.replace("<think>", "")
        if "</think>" in content:
            in_thinking = False
            content = content.replace("</think>", "")
            continue
        if in_thinking:
            continue

        print(content, end="", flush=True)
    print("\n")
    return full_response

In [100]:
query = "Explain the most importatnt Thing i should do?"
print(query)
retrieved_doc = retrieve(query)
print(retrieved_doc)
print(retrieved_doc[0])

Explain the most importatnt Thing i should do?
(['1.6 PLAN YOUR DAY\nTake five quiet minutes to review your schedule and choose the three most important tasks for the day. Writing them down clears mental clutter and gives you a clear direction before distractions arrive. Tackle the hardest or most important task first, while your willpower and focus are at their peak. A small amount of planning prevents the day from running away from you.', '1.4 MOVE YOUR BODY\nSpend 5 to 15 minutes on light physical activity such as stretching, yoga, a brisk walk, or a few bodyweight exercises. Movement increases blood flow, raises your core temperature, and releases endorphins that lift your mood. You do not need an intense workout; the goal is to wake the body gently and shake off stiffness from sleeping. Consistency matters far more than intensity.', '1.2 GET SUNLIGHT EARLY\nWithin the first 30 minutes of waking, expose yourself to natural daylight. Open the curtains, step onto a balcony, or take a

In [101]:
ask_question(query)

Q: Explain the most importatnt Thing i should do?

A: Based on the provided context, the **most important thing you should do** is **plan your day by reviewing your schedule and prioritizing the three most important tasks** (as outlined in **1.6 PLAN YOUR DAY**). Here's why:

1. **It directly prevents chaos**: The context explicitly states, *"A small amount of planning prevents the day from running away from you."* Without this step, your day risks becoming disorganized and unfocused.
2. **It sets the foundation for productivity**: By writing down the top 3 tasks (especially the hardest/important ones first), you:
   - Clear mental clutter (reducing distractions).
   - Create a clear direction for the day.
   - Leverage peak willpower and focus when starting.
3. **It is prioritized over physical activity**: While **1.4 MOVE YOUR BODY** (light exercise) is valuable for health, it is presented as a *secondary* step. The planning step is framed as the **essential first action** to structu

'Based on the provided context, the **most important thing you should do** is **plan your day by reviewing your schedule and prioritizing the three most important tasks** (as outlined in **1.6 PLAN YOUR DAY**). Here\'s why:\n\n1. **It directly prevents chaos**: The context explicitly states, *"A small amount of planning prevents the day from running away from you."* Without this step, your day risks becoming disorganized and unfocused.\n2. **It sets the foundation for productivity**: By writing down the top 3 tasks (especially the hardest/important ones first), you:\n   - Clear mental clutter (reducing distractions).\n   - Create a clear direction for the day.\n   - Leverage peak willpower and focus when starting.\n3. **It is prioritized over physical activity**: While **1.4 MOVE YOUR BODY** (light exercise) is valuable for health, it is presented as a *secondary* step. The planning step is framed as the **essential first action** to structure your day effectively.\n\n**Why not the phy

In [102]:
app = FastAPI(title="RAG API")
app.chunks = []
app.vectorize = None
app.chunk_vectors = None
app.llm = init_chat_model(model=model, model_provider="ollama", temprature = TEMPERATURE)

In [103]:
def extract_pdf_pages(pdf_content):
    pdf = pypdfium2.PdfDocument(pdf_content)
    pages = []
    for page_i in range(len(pdf)):
        page = pdf.get_page(page_i)
        textpage = page.get_textpage()
        page_text = textpage.get_text_bounded().strip()
        if page_text:
            pages.append(page_text)
    return pages


In [104]:
syllabus = extract_pdf_pages("C:\Advait\Study Material\Semester 5\BCSE355L_CLOUD-ARCHITECTURE-DESIGN_TH_1.1_79_BCSE355L.pdf")
chunks = syllabus

<>:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
<>:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
C:\Users\advai\AppData\Local\Temp\ipykernel_5624\1826791256.py:1: SyntaxWarning: "\A" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\A"? A raw string is also an option.
  syllabus = extract_pdf_pages("C:\Advait\Study Material\Semester 5\BCSE355L_CLOUD-ARCHITECTURE-DESIGN_TH_1.1_79_BCSE355L.pdf")


In [105]:
vectorize = TfidfVectorizer()
chunks_vectors = vectorize.fit_transform(syllabus)
chunks_vectors

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 369 stored elements and shape (3, 281)>

In [106]:
query = "How many modules are there"

ask_question(query)

In [107]:
def retrieve(query: str, k: int = 6):
    if not app.chunks or not app.vectorize:
        return []
    query_vector = vectorize.transform([query])
    similarity = cosine_similarity(query_vector, chunks_vectors).flatten()
    top_k_indices = np.argsort(similarity)[::-1][:k]
    return [chunks[i] for i in top_k_indices], similarity[top_k_indices]

In [108]:
def answer(query: str, k: int = 2):
    if not app.chunks:
        raise HTTPException(400, "Upload doc first")
    context_chunks, scores = retrieve(query, k)
    context_string = "\n\n".join(
        f"<context>\n{c}\n</context>" for c in context_chunks
    )
    prompt = RAG_PROMPT.format(context=context_string, query=query)
    for chunk in llm.stream(prompt):
        yield chunk.content

In [109]:
app = FastAPI()
@app.post("/upload", description="Upload the PDF file")
async def upload(file: UploadFile = File(...)):
    if not file.filename.endswith(".pdf"):
        raise HTTPException(400, "Only PDF files is allowed")
    content = await file.read()
    pages = extract_pdf_pages(content)

    if not pages:
        raise HTTPException(400, "Error in reading the pdf file")
    
    app.chunks.extend(pages)

    app.vectorize = TfidfVectorizer()
    app.chunk_vectors = app.vectorize.fit_transform(app.chunks)

    return {
        "filename" : file.filename,
        "page_added" : len(pages)
    }